# ️ Glu-Stock: 02_STRATEGIC_BRAIN
**Phase**: Strategic Intelligence (The Auditor) | v19.0 (Standard Compliance)

This notebook uses **Qwen2.5-Coder-7B-Instruct** to generate dynamic Trade Plans (Buy, TP, SL) with full strategic reasoning.

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch yfinance firebase-admin ta


In [ ]:
import json, os, firebase_admin, pandas as pd, time, re, torch, ta
from concurrent.futures import ThreadPoolExecutor
from transformers import pipeline
from firebase_admin import credentials, firestore
from datetime import datetime
from kaggle_secrets import UserSecretsClient
import yfinance as yf


In [ ]:
class StrategicBrain:
    def __init__(self, model_id="unsloth/Qwen2.5-Coder-7B-Instruct"):
        print(f"Initializing Strategic Brain ({model_id})...", flush=True)
        self.pipe = pipeline(
            "text-generation",
            model=model_id,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device_map="auto",
        )
        print("Model loaded successfully!", flush=True)
        
    def audit_signal(self, ticker, context_data, technical_metrics, current_price):
        # --- ENRICHED AUDIT PROMPT (v19.0) ---
        prompt = f"""
        SYSTEM: Institutional Financial Auditor Engine. Mode: High-Precision Reasoning.
        
        DATA INPUT FOR {ticker}:
        - Current Market Price: {current_price}
        - Volatility (ATR 14): {technical_metrics.get('atr', 'N/A')}
        - Relative Strength (RSI): {technical_metrics.get('rsi', 'N/A')}
        - Trend (MACD): {technical_metrics.get('macd', 'N/A')}
        - Volatility Band (BB_Width): {technical_metrics.get('bb_width', 'N/A')}
        
        Last 14-Day Price Action Context:
        {{context_data}}
        
        MANDATORY ANALYTICAL TASKS:
        1. CHAIN-OF-THOUGHT: Analyze technical confluence (Indicators + Price Action).
        2. DECISION: PROCEED only if setup has High Conviction. Otherwise VETO.
        3. DYNAMIC TARGETS: If PROCEED, set BUY_PRICE, TP_PRICE, and SL_PRICE.
           *Note: Suggested SL should ideally factor in 2x ATR volatility.*
        
        OUTPUT FORMAT (STRICT JSON ONLY):
        {{
            "reasoning": "Detailed logic for institutional audit documentation",
            "decision": "PROCEED" or "VETO",
            "buy_price": float,
            "tp_price": float,
            "sl_price": float
        }}
        
        EXECUTE AUDIT LOGIC NOW:
        """
        messages = [
            {{"role": "system", "content": "You are a deterministic financial logic engine. You only output valid JSON for simulator integration."}},
            {{"role": "user", "content": prompt}},
        ]
        
        try:
            outputs = self.pipe(messages, max_new_tokens=1024, do_sample=False, pad_token_id=self.pipe.tokenizer.eos_token_id)
            response = outputs[0]["generated_text"][-1]["content"]
            match = re.search(r'\{.*\}', response, re.DOTALL)
            return json.loads(match.group(0)) if match else {"decision": "VETO", "reasoning": "Parser Error"}
        except Exception as e:
            return {{"decision": "VETO", "reasoning": f"Engine Error: {str(e)}", "buy_price": 0, "tp_price": 0, "sl_price": 0}}


In [ ]:
def run_strategic_audit():
    print("--- GLU-STOCK STRATEGIC PLANNING STARTING (REASONING MODE) ---", flush=True)
    secrets = UserSecretsClient()
    cred_json = json.loads(secrets.get_secret("FIREBASE_KEY_JSON"))
    if not firebase_admin._apps: firebase_admin.initialize_app(credentials.Certificate(cred_json))
    db = firestore.client()
    
    wait_start = time.time()
    pending = []
    while time.time() - wait_start < 1800:
        pending = list(db.collection("glu_stock_queue_signals").get())
        if pending: break
        time.sleep(30)
    if not pending: return
    
    brain = StrategicBrain()
    
    def process_ticker(ticker_info):
        ticker = list(ticker_info.keys())[0]
        try:
            df = yf.download(ticker, period='60d', progress=False)
            if df.empty: return None
            
            # --- CALCULATE ADVANCED METRICS (v19.0) ---
            atr = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range().iloc[-1]
            rsi = ta.momentum.RSIIndicator(df['Close'], window=14).rsi().iloc[-1]
            macd = ta.trend.MACD(df['Close']).macd_diff().iloc[-1]
            bb = ta.volatility.BollingerBands(df['Close'])
            bb_w = ((bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()).iloc[-1]
            
            metrics = {'atr': f"{atr:.2f}", 'rsi': f"{rsi:.2f}", 'macd': f"{macd:.2f}", 'bb_width': f"{bb_w:.2%}"}
            curr_p = float(df['Close'].iloc[-1])
            plan = brain.audit_signal(ticker, df.tail(14).to_string(), metrics, curr_p)
            
            if plan.get('decision') == 'PROCEED':
                plan['metrics'] = metrics # Store for visualizer
                db.collection("glu_stock_final_signals").add({'ticker': ticker, 'plan': plan, 'timestamp': datetime.now().isoformat()})
                print(f"[PLANNED] {ticker} | Buy: {plan.get('buy_price')}")
            else:
                print(f"[VETOED] {ticker}: {plan.get('reasoning')[:50]}...")
            return ticker
        except Exception as e:
            print(f"[ERR] {ticker}: {e}")
            return None

    # Parallel Data Processing
    all_tickers_data = [doc.to_dict().get('payload', {}) for doc in pending]
    ticker_list = []
    for payload in all_tickers_data:
        for t, i in payload.items(): ticker_list.append({t: i})
        
    print(f"Auditing {len(ticker_list)} tickers using Strategic Brain...", flush=True)
    with ThreadPoolExecutor(max_workers=4) as executor:
        list(executor.map(process_ticker, ticker_list))
    
    for doc in pending: doc.reference.delete()
    db.collection("glu_stock_state").document("last_run").set({"stage": "02", "status": "done", "timestamp": datetime.now().isoformat()})

if __name__ == "__main__":
    run_strategic_audit()
